In [1]:
from pathlib import Path
import sys

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the project root containing the src/ directory")

sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import text
from src.database.connection import get_engine
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

engine = get_engine()
print("✅ Connected to Neon successfully")

✅ Connected to Neon successfully


In [3]:
query = """
SELECT 
    s.sale_id,
    s.sale_date,
    s.store_id,
    s.customer_id,
    s.total_amount,
    s.payment_method,
    si.product_id,
    si.quantity,
    si.unit_price,
    si.discount,
    p.product_name,
    p.category_id,
    p.cost_price,
    c.category_name,
    st.store_name,
    st.city AS store_city,
    st.country AS store_country
FROM sales s
JOIN sale_items si ON s.sale_id = si.sale_id
JOIN products p ON si.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
JOIN stores st ON s.store_id = st.store_id
"""

df = pd.read_sql(query, engine)
df['sale_date'] = pd.to_datetime(df['sale_date'])

print(f"عدد الصفوف: {len(df):,}")
print(f"الفترة: من {df['sale_date'].min().date()} إلى {df['sale_date'].max().date()}")
df.head()

عدد الصفوف: 2,956
الفترة: من 2024-10-13 إلى 2026-10-13


,sale_id,sale_date,store_id,customer_id,total_amount,payment_method,product_id,quantity,unit_price,discount,product_name,category_id,cost_price,category_name,store_name,store_city,store_country
0,4372,2026-06-21 04:23:47.870605+00:00,15,3297,916.98,Cash,152,2,458.49,0.00,Customer-focused attitude-oriented infrastructure,15,208.64,Pet Supplies,"Fuentes, Lewis and Norman Store",Jasonfort,Iraq
1,4373,2026-02-15 19:51:21.174506+00:00,17,1457,3649.90,Mobile Payment,429,2,326.16,0.00,De-engineered asymmetric superstructure,3,195.37,Home & Kitchen,"Peters, Henry and Robinson Store",South Seanburgh,Honduras
2,4373,2026-02-15 19:51:21.174506+00:00,17,1457,3649.90,Mobile Payment,187,1,561.40,0.00,Pre-emptive context-sensitive productivity,13,298.22,Shoes,"Peters, Henry and Robinson Store",South Seanburgh,Honduras
3,4373,2026-02-15 19:51:21.174506+00:00,17,1457,3649.90,Mobile Payment,56,3,105.13,14.22,Function-based dynamic application,14,63.60,Garden,"Peters, Henry and Robinson Store",South Seanburgh,Honduras
4,4373,2026-02-15 19:51:21.174506+00:00,17,1457,3649.90,Mobile Payment,6,4,500.20,0.62,Centralized intangible support,10,258.78,Automotive,"Peters, Henry and Robinson Store",South Seanburgh,Honduras


In [4]:
df['year'] = df['sale_date'].dt.year
df['month'] = df['sale_date'].dt.month
df['day'] = df['sale_date'].dt.day
df['day_of_week'] = df['sale_date'].dt.dayofweek          # 0 = Monday
df['day_name'] = df['sale_date'].dt.day_name()
df['week_of_year'] = df['sale_date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['sale_date'].dt.quarter
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_month_start'] = df['sale_date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['sale_date'].dt.is_month_end.astype(int)

print("✅ تم إنشاء الميزات الزمنية")
df[['sale_date', 'day_name', 'is_weekend', 'month', 'quarter']].head(10)

✅ تم إنشاء الميزات الزمنية


,sale_date,day_name,is_weekend,month,quarter
0,2026-06-21 04:23:47.870605+00:00,Sunday,1,6,2
1,2026-02-15 19:51:21.174506+00:00,Sunday,1,2,1
2,2026-02-15 19:51:21.174506+00:00,Sunday,1,2,1
3,2026-02-15 19:51:21.174506+00:00,Sunday,1,2,1
4,2026-02-15 19:51:21.174506+00:00,Sunday,1,2,1
5,2026-02-15 19:51:21.174506+00:00,Sunday,1,2,1
6,2024-11-23 21:49:51.601489+00:00,Saturday,1,11,4
7,2025-12-19 10:25:02.804979+00:00,Friday,0,12,4
8,2025-05-26 09:43:08.277711+00:00,Monday,0,5,2
9,2025-05-26 09:43:08.277711+00:00,Monday,0,5,2


In [5]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'] / 100)
df['cost'] = df['quantity'] * df['cost_price']
df['profit'] = df['revenue'] - df['cost']
df['profit_margin'] = np.where(df['revenue'] > 0, df['profit'] / df['revenue'], 0)
df['has_discount'] = (df['discount'] > 0).astype(int)

print("✅ تم حساب الإيرادات والربح")
df[['product_name', 'quantity', 'unit_price', 'discount', 'revenue', 'profit', 'profit_margin']].head(10)

✅ تم حساب الإيرادات والربح


,product_name,quantity,unit_price,discount,revenue,profit,profit_margin
0,Customer-focused attitude-oriented infrastructure,2,458.49,0.00,916.980000,499.700000,0.544941
1,De-engineered asymmetric superstructure,2,326.16,0.00,652.320000,261.580000,0.401000
2,Pre-emptive context-sensitive productivity,1,561.40,0.00,561.400000,263.180000,0.468792
3,Function-based dynamic application,3,105.13,14.22,270.541542,79.741542,0.294748
4,Centralized intangible support,4,500.20,0.62,1988.395040,953.275040,0.479419
5,Persevering methodical interface,3,59.08,0.00,177.240000,81.240000,0.458362
6,Balanced high-level knowledge user,1,46.46,0.00,46.460000,26.360000,0.567370
7,Adaptive intermediate orchestration,1,171.71,0.00,171.710000,93.820000,0.546386
8,Focused even-keeled definition,1,569.65,0.00,569.650000,275.210000,0.483121
9,Total fresh-thinking paradigm,3,69.93,0.00,209.790000,109.560000,0.522237


In [6]:
daily = (
    df.groupby(['product_id', 'store_id', 'sale_date'])
    .agg(
        quantity_sold=('quantity', 'sum'),
        revenue=('revenue', 'sum'),
        num_transactions=('sale_id', 'nunique'),
        avg_discount=('discount', 'mean'),
        avg_unit_price=('unit_price', 'mean')
    )
    .reset_index()
)

# إضافة معلومات المنتج والمتجر
product_info = df[['product_id', 'product_name', 'category_id', 'category_name']].drop_duplicates()
store_info = df[['store_id', 'store_name', 'store_city', 'store_country']].drop_duplicates()

daily = daily.merge(product_info, on='product_id', how='left')
daily = daily.merge(store_info, on='store_id', how='left')

print(f"✅ تم إنشاء {len(daily):,} صف من الطلب اليومي")
daily.head(10)

✅ تم إنشاء 2,956 صف من الطلب اليومي


,product_id,store_id,sale_date,quantity_sold,revenue,num_transactions,avg_discount,avg_unit_price,product_name,category_id,category_name,store_name,store_city,store_country
0,1,3,2025-11-28 10:38:03.277856+00:00,1,85.550105,1,13.05,98.39,Progressive discrete implementation,5,Sports,Obrien-Carson Store,Port Mitchell,Paraguay
1,1,3,2025-12-07 16:28:59.688785+00:00,2,196.780000,1,0.00,98.39,Progressive discrete implementation,5,Sports,Obrien-Carson Store,Port Mitchell,Paraguay
2,1,12,2026-09-29 20:25:55.240250+00:00,3,295.170000,1,0.00,98.39,Progressive discrete implementation,5,Sports,"Li, Melton and Smith Store",Susanville,Suriname
3,1,14,2025-03-02 14:23:20.322665+00:00,4,393.560000,1,0.00,98.39,Progressive discrete implementation,5,Sports,Garza-Henderson Store,North Mistyside,Greenland
4,1,15,2025-12-01 10:41:23.710006+00:00,3,295.170000,1,0.00,98.39,Progressive discrete implementation,5,Sports,"Fuentes, Lewis and Norman Store",Jasonfort,Iraq
5,1,20,2025-01-12 21:20:51.129269+00:00,2,196.780000,1,0.00,98.39,Progressive discrete implementation,5,Sports,Stone-Ward Store,East Johnmouth,Spain
6,2,4,2026-07-20 13:40:50.148050+00:00,2,410.700000,1,0.00,205.35,Business-focused zero administration frame,3,Home & Kitchen,Collins-Mitchell Store,Port Julie,Reunion
7,2,12,2025-08-21 11:51:28.292748+00:00,2,410.700000,1,0.00,205.35,Business-focused zero administration frame,3,Home & Kitchen,"Li, Melton and Smith Store",Susanville,Suriname
8,2,13,2025-04-07 01:15:57.847270+00:00,1,205.350000,1,0.00,205.35,Business-focused zero administration frame,3,Home & Kitchen,Whitehead-Hawkins Store,Gomezshire,Sri Lanka
9,2,17,2026-09-12 05:20:22.018256+00:00,4,715.603680,1,12.88,205.35,Business-focused zero administration frame,3,Home & Kitchen,"Peters, Henry and Robinson Store",South Seanburgh,Honduras


In [7]:
daily = daily.sort_values(['product_id', 'store_id', 'sale_date']).copy()

group = daily.groupby(['product_id', 'store_id'])

# Lag Features
daily['lag_1'] = group['quantity_sold'].shift(1)
daily['lag_7'] = group['quantity_sold'].shift(7)
daily['lag_14'] = group['quantity_sold'].shift(14)
daily['lag_30'] = group['quantity_sold'].shift(30)

# Rolling Features
daily['rolling_mean_7'] = group['quantity_sold'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)
daily['rolling_mean_14'] = group['quantity_sold'].transform(
    lambda x: x.shift(1).rolling(14, min_periods=1).mean()
)
daily['rolling_std_7'] = group['quantity_sold'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).std()
)

print("✅ تم إنشاء Lag & Rolling Features")
daily[['product_id', 'store_id', 'sale_date', 'quantity_sold', 'lag_1', 'lag_7', 'rolling_mean_7']].head(15)

✅ تم إنشاء Lag & Rolling Features


,product_id,store_id,sale_date,quantity_sold,lag_1,lag_7,rolling_mean_7
0,1,3,2025-11-28 10:38:03.277856+00:00,1,NaN,NaN,NaN
1,1,3,2025-12-07 16:28:59.688785+00:00,2,1.0,NaN,1.0
2,1,12,2026-09-29 20:25:55.240250+00:00,3,NaN,NaN,NaN
3,1,14,2025-03-02 14:23:20.322665+00:00,4,NaN,NaN,NaN
4,1,15,2025-12-01 10:41:23.710006+00:00,3,NaN,NaN,NaN
5,1,20,2025-01-12 21:20:51.129269+00:00,2,NaN,NaN,NaN
6,2,4,2026-07-20 13:40:50.148050+00:00,2,NaN,NaN,NaN
7,2,12,2025-08-21 11:51:28.292748+00:00,2,NaN,NaN,NaN
8,2,13,2025-04-07 01:15:57.847270+00:00,1,NaN,NaN,NaN
9,2,17,2026-09-12 05:20:22.018256+00:00,4,NaN,NaN,NaN


In [8]:
daily['year'] = daily['sale_date'].dt.year
daily['month'] = daily['sale_date'].dt.month
daily['day_of_week'] = daily['sale_date'].dt.dayofweek
daily['is_weekend'] = daily['day_of_week'].isin([5, 6]).astype(int)
daily['quarter'] = daily['sale_date'].dt.quarter

print("✅ الميزات النهائية جاهزة")
print(f"عدد الصفوف: {len(daily):,}")
print(f"عدد الأعمدة: {len(daily.columns)}")
print("\nالأعمدة:")
print(daily.columns.tolist())

✅ الميزات النهائية جاهزة
عدد الصفوف: 2,956
عدد الأعمدة: 26

الأعمدة:
['product_id', 'store_id', 'sale_date', 'quantity_sold', 'revenue', 'num_transactions', 'avg_discount', 'avg_unit_price', 'product_name', 'category_id', 'category_name', 'store_name', 'store_city', 'store_country', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'year', 'month', 'day_of_week', 'is_weekend', 'quarter']


In [9]:
print("📈 إحصائيات الميزات الرئيسية:\n")
print(daily[['quantity_sold', 'revenue', 'lag_1', 'lag_7', 'rolling_mean_7', 'avg_discount']].describe())

📈 إحصائيات الميزات الرئيسية:

       quantity_sold      revenue       lag_1  lag_7  rolling_mean_7  avg_discount
count    2956.000000  2956.000000  412.000000    0.0      412.000000   2956.000000
mean        2.492219   708.292977    2.495146    NaN        2.485841      2.206255
std         1.120010   572.753199    1.095212    NaN        1.076991      4.137934
min         1.000000    15.770754    1.000000    NaN        1.000000      0.000000
25%         1.000000   267.255000    2.000000    NaN        2.000000      0.000000
50%         3.000000   543.249413    3.000000    NaN        2.500000      0.000000
75%         3.000000  1018.280000    3.000000    NaN        3.000000      2.322500
max         4.000000  2871.880000    4.000000    NaN        4.000000     15.000000


In [10]:
import os
os.makedirs("data", exist_ok=True)

# حفظ Parquet (أسرع)
daily.to_parquet("data/features_demand.parquet", index=False)
print("💾 تم الحفظ: data/features_demand.parquet")

# حفظ CSV أيضًا
daily.to_csv("data/features_demand.csv", index=False)
print("💾 تم الحفظ: data/features_demand.csv")

print("\n🎉 Feature Engineering اكتمل بنجاح!")

💾 تم الحفظ: data/features_demand.parquet
💾 تم الحفظ: data/features_demand.csv

🎉 Feature Engineering اكتمل بنجاح!
